In [106]:
import pandas as pd
import matplotlib.pyplot as plt
import glob

In [107]:
# data klasöründeki parquet dosyalarını alıyorum
files = glob.glob("../data/*.parquet")
print("Number of files:", len(files))
files[:3]

Number of files: 12


['../data/yellow_tripdata_2015-10.parquet',
 '../data/yellow_tripdata_2015-09.parquet',
 '../data/yellow_tripdata_2015-08.parquet']

In [108]:
# İlk olarak sadece Ocak ayı verisini inceliyorum
df_january = pd.read_parquet("../data/yellow_tripdata_2015-01.parquet")
df_january.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2015-01-01 00:11:33,2015-01-01 00:16:48,1,1.0,1,N,41,166,1,5.7,0.5,0.5,1.40,0.0,0.0,8.40,None,None
1,1,2015-01-01 00:18:24,2015-01-01 00:24:20,1,0.9,1,N,166,238,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,None,None
2,1,2015-01-01 00:26:19,2015-01-01 00:41:06,1,3.5,1,N,238,162,1,13.2,0.5,0.5,2.90,0.0,0.0,17.40,None,None
3,1,2015-01-01 00:45:26,2015-01-01 00:53:20,1,2.1,1,N,162,263,1,8.2,0.5,0.5,2.37,0.0,0.0,11.87,None,None
4,1,2015-01-01 00:59:21,2015-01-01 01:05:24,1,1.0,1,N,236,141,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,None,None


# Veriye genel bakış


In [109]:
# satır ve sütun sayısının kontrol edilmesi
df_january.shape

(12741035, 19)

In [110]:
# sütun isimlerini kontrol ediyorum
df_january.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee'],
      dtype='str')

In [111]:
# veri tiplerinin kontrol edilmesi
df_january.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                   int64
trip_distance                   float64
RatecodeID                        int64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge             object
airport_fee                      object
dtype: object

In [112]:
# Eksik değerleri kontrol ediyorum
df_january.isnull().sum()

VendorID                        0
tpep_pickup_datetime            0
tpep_dropoff_datetime           0
passenger_count                 0
trip_distance                   0
RatecodeID                      0
store_and_fwd_flag              0
PULocationID                    0
DOLocationID                    0
payment_type                    0
fare_amount                     0
extra                           0
mta_tax                         0
tip_amount                      0
tolls_amount                    0
improvement_surcharge           3
total_amount                    0
congestion_surcharge     12741035
airport_fee              12741035
dtype: int64

In [113]:
# Boş sütunları kaldır
df = df.dropna(axis=1, how="all")
print("Remaining columns:", len(df.columns))

Remaining columns: 4


In [114]:
df.columns # 'congestion_surcharge' ve 'airport_fee' sütunları kaldırıldı

Index(['tpep_pickup_datetime', 'passenger_count', 'trip_distance',
       'fare_amount'],
      dtype='str')

In [115]:
df.head()

,tpep_pickup_datetime,passenger_count,trip_distance,fare_amount
0,2015-10-01 00:04:43,1,1.6,7.0
1,2015-10-01 00:13:58,2,10.2,34.0
2,2015-10-01 00:53:57,1,1.4,6.0
3,2015-10-01 00:59:04,1,0.4,3.5
4,2015-10-01 00:01:45,1,2.0,8.5


In [116]:
print("Dataset shape:", df.shape)

Dataset shape: (146039231, 4)


In [117]:
# pickup sütununu datetime türüne dönüştürülmesi
df["tpep_pickup_datetime"] = pd.to_datetime(
    df["tpep_pickup_datetime"]
)
# saatlik ytrip sayısı hesaplanması
january_hourly = (
    df_january.set_index("tpep_pickup_datetime")
    .resample("h")
    .size()
    .reset_index(name="trip_count")
)
january_hourly.head()

,tpep_pickup_datetime,trip_count
0,2015-01-01 00:00:00,28312
1,2015-01-01 01:00:00,31707
2,2015-01-01 02:00:00,28068
3,2015-01-01 03:00:00,24288
4,2015-01-01 04:00:00,17081


### yıl boyu trip sayısı hesaplanması

In [118]:
# Tüm aylık dosyalardan gerekli sütunları okuyorum
all_data = []

for file in files:
    print("Reading:", file)
    temp_df = pd.read_parquet(
        file,
        columns=[
            "tpep_pickup_datetime",
            "passenger_count",
            "trip_distance",
            "fare_amount"
        ]
    )
    all_data.append(temp_df)
# Tüm ayları tek dataframe içinde birleştiriyorum
df = pd.concat(all_data, ignore_index=True)
print("Full dataset shape:", df.shape)

Reading: ../data/yellow_tripdata_2015-10.parquet
Reading: ../data/yellow_tripdata_2015-09.parquet
Reading: ../data/yellow_tripdata_2015-08.parquet
Reading: ../data/yellow_tripdata_2015-11.parquet
Reading: ../data/yellow_tripdata_2015-01.parquet
Reading: ../data/yellow_tripdata_2015-03.parquet
Reading: ../data/yellow_tripdata_2015-12.parquet
Reading: ../data/yellow_tripdata_2015-02.parquet
Reading: ../data/yellow_tripdata_2015-07.parquet
Reading: ../data/yellow_tripdata_2015-06.parquet
Reading: ../data/yellow_tripdata_2015-04.parquet
Reading: ../data/yellow_tripdata_2015-05.parquet
Full dataset shape: (146039231, 4)


In [119]:
# Sayısal sütunların temel istatistiklerini inceliyorum
df.describe()

,tpep_pickup_datetime,passenger_count,trip_distance,fare_amount
count,146039231,1.460392e+08,1.460392e+08,1.460392e+08
mean,2015-06-26 17:29:12.342668,1.680858e+00,1.184584e+01,1.293904e+01
min,2015-01-01 00:00:00,0.000000e+00,-4.084012e+07,-4.960000e+02
25%,2015-03-27 13:26:37,1.000000e+00,1.000000e+00,6.500000e+00
50%,2015-06-20 18:28:43,1.000000e+00,1.710000e+00,9.500000e+00
75%,2015-09-26 23:56:36,2.000000e+00,3.200000e+00,1.450000e+01
max,2015-12-31 23:59:59,9.000000e+00,5.901661e+07,8.259986e+05
std,NaN,1.333530e+00,1.093597e+04,1.243789e+02


In [120]:
# sayısal sütunların min, max, mean ve median değerlerini kontrol ediyorum
numeric_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount"
]

for col in numeric_columns:
    print("\nColumn:", col)
    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())
    print("Mean:", df[col].mean())
    print("Median:", df[col].median())


Column: passenger_count
Minimum: 0
Maximum: 9
Mean: 1.6808580976436394
Median: 1.0

Column: trip_distance
Minimum: -40840124.4
Maximum: 59016609.3
Mean: 11.845839219462887
Median: 1.71

Column: fare_amount
Minimum: -496.0
Maximum: 825998.61
Mean: 12.939035017035936
Median: 9.5


In [121]:
# mantıksız olabilecek değerleri kontrol edilmesi
print("Passenger count <= 0:", (df["passenger_count"] <= 0).sum())
print("Trip distance <= 0:", (df["trip_distance"] <= 0).sum())
print("Fare amount <= 0:", (df["fare_amount"] <= 0).sum())

print("Very long trips > 100 miles:", (df["trip_distance"] > 100).sum())
print("Very high fares > 500 dollars:", (df["fare_amount"] > 500).sum())

Passenger count <= 0: 40683
Trip distance <= 0: 876729
Fare amount <= 0: 87046
Very long trips > 100 miles: 2325
Very high fares > 500 dollars: 382


In [122]:
# Mantıksız değerleri temizliyorum
df_clean = df[
    (df["passenger_count"] > 0) &
    (df["passenger_count"] <= 6) &
    (df["trip_distance"] > 0) &
    (df["trip_distance"] <= 100) &
    (df["fare_amount"] > 0) &
    (df["fare_amount"] <= 500)
]

df_clean = df_clean.copy()

In [123]:
# # pickup datetime sütununu datetime formatına dönüştürülmesi
df_clean["tpep_pickup_datetime"] = pd.to_datetime(
    df_clean["tpep_pickup_datetime"],
    errors="coerce"
)

# boş datetime değerlerini kaldırıyorum
df_clean = df_clean.dropna(subset=["tpep_pickup_datetime"])

# sadece 2015 yılı içindeki verileri tutuyorum
df_clean = df_clean[
    (df_clean["tpep_pickup_datetime"] >= "2015-01-01") &
    (df_clean["tpep_pickup_datetime"] < "2016-01-01")
]
print("Cleaned dataset shape:", df_clean.shape)

Cleaned dataset shape: (145060992, 4)


In [124]:
# temizlik sonrası temel istatistikleri tekrar kontrol ediyorum
df_clean.describe()

,tpep_pickup_datetime,passenger_count,trip_distance,fare_amount
count,145060992,1.450610e+08,1.450610e+08,1.450610e+08
mean,2015-06-26 17:48:27.213775,1.683120e+00,2.989843e+00,1.286644e+01
min,2015-01-01 00:00:00,1.000000e+00,1.000000e-02,1.000000e-02
25%,2015-03-27 13:42:53,1.000000e+00,1.020000e+00,6.500000e+00
50%,2015-06-20 18:51:55,1.000000e+00,1.740000e+00,9.500000e+00
75%,2015-09-27 00:13:22,2.000000e+00,3.200000e+00,1.450000e+01
max,2015-12-31 23:59:59,6.000000e+00,1.000000e+02,5.000000e+02
std,NaN,1.335038e+00,3.617380e+00,1.059236e+01


## Veri setinde bazı mantıksız değerler tespit ettim.
- `trip_distance` sütununda negatif ve çok büyük değerler vardı.
- `fare_amount` sütununda negatif ve gerçekçi olmayan değerler vardı.
- `passenger_count` sütununda 0 yolcu olan kayıtlar vardı.
Bu nedenle saatlik trip count oluşturmadan önce veri temizliğ yaptım.


### Yıl boyu saatlik trip sayısı

In [125]:
# temizlenmiş veri ile saatlik trip sayısını hesaplıyorum
hourly_data = (
    df_clean.set_index("tpep_pickup_datetime")
    .resample("h")
    .size()
    .reset_index(name="trip_count")
)
hourly_data.head()

,tpep_pickup_datetime,trip_count
0,2015-01-01 00:00:00,28093
1,2015-01-01 01:00:00,31432
2,2015-01-01 02:00:00,27763
3,2015-01-01 03:00:00,24007
4,2015-01-01 04:00:00,16806


In [126]:
# saatlik veri boyutunu kontrol edilmesi
hourly_data.shape

(8760, 2)

In [127]:
# saatlik trip count verisini csv olarak kaydediyorum
hourly_data.to_csv(
    "../outputs/hourly_trip_counts_2015.csv",
    index=False
)
print("CSV dosyası kaydedildi")

CSV dosyası kaydedildi
